# Example 10.1.3 / 4: Keep a Pure Network When Possible

Companion notebook for **10.1.3 Multicommodity Flows and Side Constraints** in the AMPL lab Documentation, section "Keep a Pure Network When Possible". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Keep a Pure Network When Possible

Not every extra limit needs a side constraint. The book's example is a throughput limit at a city. Written as `supply[k] + sum {(i, k) in LINKS} Ship[i, k] <= city_cap[k]`, it adds a third coefficient to every incoming flow. Instead, split each city into a `Supply` node, which receives inflow and supply, and a `Demand` node, which sends outflow and meets demand. A `Through` arc from `Supply[k]` to `Demand[k]` carries the city's throughput, and its upper bound is the limit:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CITIES;  # network nodes
set LINKS within CITIES cross CITIES;  # directed shipping links

param supply {CITIES} >= 0 default 0;   # units available
param demand {CITIES} >= 0 default 0;   # units required
# balanced network
check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];
param cost {LINKS} >= 0;                # cost per unit shipped
param city_cap {CITIES} >= 0;           # max throughput at a city
param link_cap {LINKS} >= 0;            # max shipment on a link

minimize TotalCost;  # terms are added by the arc declarations

node Supply {k in CITIES}: net_out = supply[k];  # receives inflow and supply
node Demand {k in CITIES}: net_in = demand[k];  # sends outflow and meets demand

# shipment from city i to city j
arc Ship {(i, j) in LINKS} >= 0, <= link_cap[i, j],
    from Demand[i], to Supply[j], obj TotalCost cost[i, j];

# throughput of city k; its bound is the limit
arc Through {k in CITIES} >= 0, <= city_cap[k],
    from Supply[k], to Demand[k];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["CITIES"] = ["plant", "hub", "east"]
ampl.set["LINKS"] = [("plant", "hub"), ("hub", "east"), ("plant", "east")]
ampl.param["supply"] = {"plant": 5}
ampl.param["demand"] = {"east": 5}
ampl.param["cost"] = {("plant", "hub"): 1, ("hub", "east"): 1, ("plant", "east"): 4}
ampl.param["city_cap"] = {"plant": 5, "hub": 3, "east": 5}
ampl.param["link_cap"] = {("plant", "hub"): 5, ("hub", "east"): 5, ("plant", "east"): 5}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.get_data("Ship").to_dict())
print(ampl.get_data("Through").to_dict())

HiGHS 1.14.0: optimal solution; objective 14
0 simplex iterations
0 barrier iterations
solved
{('hub', 'east'): 3, ('plant', 'east'): 2, ('plant', 'hub'): 3}
{'east': 5, 'hub': 3, 'plant': 5}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set CITIES := plant hub east;
set LINKS := (plant,hub) (hub,east) (plant,east);
param supply := plant 5;
param demand := east 5;
param cost := [plant,hub] 1 [hub,east] 1 [plant,east] 4;
param city_cap := plant 5 hub 3 east 5;
param link_cap := [plant,hub] 5 [hub,east] 5 [plant,east] 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Ship, Through;

HiGHS 1.14.0: optimal solution; objective 14
0 simplex iterations
0 barrier iterations
Ship :=
hub   east   3
plant east   2
plant hub    3
;

Through [*] :=
 east  5
  hub  3
plant  5
;

